# Medical RAG - Systematic Build-up

This notebook demonstrates a Retrieval Augmented Generation (RAG) pipeline for medical questions, incorporating query rewriting and reranking with a similarity threshold to determine when to directly pass the query to an LLM.

## 1. Install Dependencies

First, we need to install the necessary libraries for sentence embeddings, FAISS for efficient similarity search, and datasets for loading our medical data.

In [1]:
!pip install -q sentence-transformers faiss-cpu datasets

## 2. Load Medical Dataset

We will load the MedQuAD dataset from Hugging Face, which contains question-answer pairs related to medical topics. We'll use a subset of the data for quicker processing during this demonstration.

In [2]:
from datasets import load_dataset

# Load the dataset
ds = load_dataset("AnonymousSub/MedQuAD_47441_Question_Answer_Pairs")
data = ds["train"]  # Access the training split

print(f"Total samples in dataset: {len(data)}")
print("Example data entry:")
print(data[0])

# Define column names (adjust if your dataset uses different names)
QUESTION_COL = "Questions"
ANSWER_COL = "Answers"

# Use a smaller sample size for faster execution in a demo
SAMPLE_SIZE = 3000
subset = data.select(range(min(SAMPLE_SIZE, len(data))))

# Extract documents (answers) and questions
documents = [row[ANSWER_COL] for row in subset]
questions = [row[QUESTION_COL] for row in subset]

print(f"\nUsing a subset of {len(documents)} documents for the demo.")

README.md:   0%|          | 0.00/421 [00:00<?, ?B/s]

data/train-00000-of-00001-4401d00b2bdd18(…): reconstructing file:   0%|          |  0.00B / 9.26MB            

data/train-00000-of-00001-4401d00b2bdd18(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/47441 [00:00<?, ? examples/s]

Total samples in dataset: 47441
Example data entry:
{'Questions': 'What is (are) Hepatitis B: What Asian and Pacific Islander Americans Need to Know ?', 'Answers': 'Hepatitis B is a liver disease spread through contact with blood, semen, or other body fluids from a person infected with the hepatitis B virus. The disease is most commonly spread from an infected mother to her infant at birth. Hepatitis B is also spread through sex, wound-to-wound contact, and contact with items that may have blood on them, such as shaving razors, toothbrushes, syringes, and tattoo and body piercing needles.\n                \nHepatitis B is not spread through casual contact such as shaking hands or hugging; nor is it spread by sharing food or beverages, by sneezing and coughing, or through breastfeeding.'}

Using a subset of 3000 documents for the demo.


## 3. Document Type Tagging (Simple Heuristic)

To demonstrate filtering, we'll create a simple function to 'guess' the type of medical question based on keywords. This could be replaced with a more sophisticated classification model in a real-world scenario.

In [3]:
def guess_type(q):
    q = q.lower()
    if q.startswith("what is") or q.startswith("what are"):
        return "definition"
    if "treat" in q or "cure" in q or "medicine" in q:
        return "treatment"
    if "when" in q or "doctor" in q or "emergency" in q:
        return "when_to_see_doctor"
    return "cause"

doc_types = [guess_type(q) for q in questions]

print("First 5 document types:", doc_types[:5])

First 5 document types: ['definition', 'definition', 'cause', 'definition', 'cause']


## 4. Build Embeddings and FAISS Index

We'll use a Sentence Transformer model (`all-MiniLM-L6-v2`) to convert our medical documents into numerical embeddings. These embeddings allow us to perform fast similarity searches using a FAISS index.

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Initialize the embedder model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Encode all documents into embeddings
print(f"Encoding {len(documents)} documents...")
doc_embeddings = embedder.encode(documents, show_progress_bar=True)
print("Encoding complete.")

# Build a FAISS index for efficient similarity search
# IndexFlatL2 is a simple index that stores all vectors and uses L2 distance (Euclidean distance)
index = faiss.IndexFlatL2(doc_embeddings.shape[1])
index.add(np.array(doc_embeddings)) # FAISS expects numpy arrays

print(f"FAISS index built with {index.ntotal} vectors.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 3000 documents...


Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Encoding complete.
FAISS index built with 3000 vectors.


## 5. Load Cross-Encoder Reranker

A cross-encoder model (`cross-encoder/ms-marco-MiniLM-L-6-v2`) will be used to rerank the initial set of retrieved documents. Cross-encoders are generally more accurate for relevance scoring than bi-encoders (like the embedder) because they jointly encode the query and document, allowing for deeper interaction.

In [5]:
from sentence_transformers import CrossEncoder

# Load the cross-encoder reranker model
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

print("Cross-encoder reranker loaded.")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder reranker loaded.


## 6. Query Rewrite Function

This simple rule-based function rewrites the user's query to make it more suitable for a medical search. In a production system, this step would typically involve an LLM (e.g., Claude, GPT) to perform more sophisticated query expansion or rephrasing.

In [6]:
def rewrite_query(user_query):
    # A placeholder for a more advanced LLM-based query rewrite
    return f"possible medical causes and next steps for symptoms: {user_query}"

print("Query rewrite function defined.")

Query rewrite function defined.


## 7. Retrieval and Reranking Pipeline with Similarity Threshold

This is the core RAG pipeline. It performs the following steps:
1.  **Rewrite Query**: Enhances the user's initial query.
2.  **Vector Search**: Retrieves a set of candidate documents using the FAISS index.
3.  **Filter Candidates**: Removes documents based on predefined types (e.g., exclude 'definition' types if not relevant).
4.  **Rerank**: Uses a cross-encoder to accurately score the relevance of candidate documents to the original query.
5.  **Similarity Threshold**: Filters reranked documents based on a `SIMILARITY_THRESHOLD`. If no documents meet this threshold, the function indicates that the query should be passed directly to an LLM without retrieved context.

In [7]:
SIMILARITY_THRESHOLD = 0.5  # Adjust this value based on your needs (0 to 1, higher means stricter)

def search(user_query, top_k_retrieve=6, top_k_final=3, exclude_types=None):
    exclude_types = exclude_types or []

    # Step 1: Rewrite query for better search effectiveness
    search_query = rewrite_query(user_query)

    # Step 2: Embed the rewritten query and perform vector search
    query_vec = embedder.encode([search_query])
    # Retrieve more documents than ultimately needed for reranking
    distances, indices = index.search(np.array(query_vec), top_k_retrieve)

    # Step 3: Filter out documents based on excluded types
    candidates = [
        (documents[i], doc_types[i]) for i in indices[0]
        if doc_types[i] not in exclude_types
    ]

    if not candidates:
        # No candidates even before reranking, pass to LLM directly
        return [], True

    # Step 4: Rerank candidates with a cross-encoder for improved relevance
    pairs = [(user_query, doc) for doc, _ in candidates]
    scores = reranker.predict(pairs)

    # Sort candidates by reranker score in descending order
    ranked = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)

    final_results = []
    pass_to_llm_directly = True # Assume direct LLM pass until a good chunk is found

    # Step 5: Apply similarity threshold and collect final results
    for (doc, _), score in ranked[:top_k_final]:
        if score >= SIMILARITY_THRESHOLD:
            final_results.append(doc)
            pass_to_llm_directly = False # At least one chunk meets the threshold
        else:
            # If a chunk is below threshold, and it's sorted, subsequent chunks will also be low
            break

    return final_results, pass_to_llm_directly

print("Search function defined with similarity threshold logic.")

Search function defined with similarity threshold logic.


## 8. Test the RAG Pipeline

Let's test our `search` function with an example query and see how it behaves with the similarity threshold. We will check if it retrieves relevant chunks or recommends passing the query directly to an LLM.

In [8]:
query = "I have a persistent cough and shortness of breath."
# Try a query that might not have strong matches
# query = "What is the meaning of life?"

results, pass_to_llm_directly = search(query, exclude_types=["definition"])

print("--- Query Result ---")
print(f"Original Query: {query}")

if pass_to_llm_directly:
    print("\nDecision: No highly relevant chunks found (below similarity threshold).")
    print("Action: Pass the original query directly to the LLM for a general answer.")
    # Here, you would typically make an API call to your chosen LLM with 'query'.
    # Example: llm_response = your_llm_model.generate(query)
    # print(llm_response)
else:
    print("\nDecision: Relevant chunks found above the similarity threshold.")
    print("Action: Use these chunks as context for the LLM to generate an answer.")
    print("\nTop retrieved chunks:")
    for i, r in enumerate(results, 1):
        print(f"{i}. {r}\n")

print("--------------------")

--- Query Result ---
Original Query: I have a persistent cough and shortness of breath.

Decision: No highly relevant chunks found (below similarity threshold).
Action: Pass the original query directly to the LLM for a general answer.
--------------------


## 9. (Optional) Integrate with an LLM for Final Answer Generation

This section shows how you would typically integrate the retrieved `results` (or the original `query` if `pass_to_llm_directly` is True) with a Large Language Model to generate a comprehensive, patient-friendly answer. This part is commented out as it requires an actual LLM API setup.

In [13]:
import requests

URL = "https://abhishekprajapti66--medical-chatbot-medicalchatbot-chat.modal.run"

# Prepare the data for the API request using the existing 'query' variable
request_payload = {
    "question": query
}

print(f"Sending query to chatbot API: {query}")

try:
    response = requests.post(
        URL,
        json=request_payload
    )

    print(f"\nAPI Response Status Code: {response.status_code}")
    if response.status_code == 200:
        response_data = response.json()
        if 'answer' in response_data:
            print("Chatbot Answer:")
            print(response_data['answer'])
        else:
            print("API Response Body (no 'answer' key found):")
            print(response_data)
    else:
        print(f"Error: {response.text}")
except requests.exceptions.RequestException as e:
    print(f"An error occurred during the API request: {e}")

Sending query to chatbot API: I have a persistent cough and shortness of breath.

API Response Status Code: 200
Chatbot Answer:
Hello, welcome to AskAi! I understand your concern regarding your persistent cough and shortness of breath. Coughing and shortness of breath can be caused by various respiratory infections such as influenza, pneumonia, or bronchitis. In this case, you may need to take some precautions to prevent further spread of the infection. Here are some tips:

1. Wash your hands frequently with soap and water for at least 20 seconds. This will help to reduce the risk of catching an infection.

2. Avoid touching your face, especially your mouth, nose, and eyes.

3. Stay home if you feel unwell


### Set up Hugging Face Token (Optional, but Recommended)

To avoid the warning and potentially benefit from higher rate limits and faster downloads, you can set up a Hugging Face token.

1.  **Get an HF Token**: Go to [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) and create a new token. Make sure it has at least 'read' access.
2.  **Store in Colab Secrets**: In Google Colab, click on the '🔑' icon (Secrets) in the left sidebar. Add a new secret named `HF_TOKEN` and paste your Hugging Face token as its value.
3.  **Enable access**: Make sure to toggle on 'Notebook access' for this secret in the Colab secrets panel.

In [ ]:
# Load the Hugging Face API key from Colab secrets
from google.colab import userdata
import os

try:
    os.environ["HF_TOKEN"] = userdata.get('HF_API_KEY')
    print("Hugging Face API key loaded successfully from Colab secrets.")
except userdata.SecretNotFoundError:
    print("HF_API_KEY not found in Colab secrets. Please add it and enable 'Notebook access' if you want to suppress the warning.")
except Exception as e:
    print(f"An unexpected error occurred while loading HF_API_KEY: {e}")


Hugging Face API key loaded successfully from Colab secrets.


After running the above cell and ensuring your `HF_TOKEN` is correctly set in Colab secrets, you might need to re-run the cell that produced the warning (`efbxrebS_EOV`) for the change to take effect and for the warning to disappear.